# ⚙️ Notebook 2 — Feature Engineering
**Input:** `preprocessed.csv`
**Output:** `features.csv`

Engineers 43 structured features across 5 groups:
- A. Time-of-day (cyclic + binary windows)
- B. Calendar (cyclic month, day-of-week, year offset)
- C. Meeting metadata (duration, density, meeting type)
- D. Topic category one-hot
- E. Interaction features

These features feed XGBoost and Logistic Regression.  
RobBERT-derived sentiment/tone features are added in Notebook 3.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})
BLUE = '#2B5797'

df = pd.read_csv("preprocessed.csv")
df['Topic_date'] = pd.to_datetime(df['Topic_date'], errors='coerce')
print(f"Loaded {len(df)} rows, {df.shape[1]} columns")


## A. Time-of-Day Features

In [ ]:
TOD_ORDER = {
    'Early morning': 0, 'Late morning': 1, 'Early afternoon': 2,
    'Late afternoon': 3, 'Evening': 4, 'Night': 5, 'Unknown': -1
}

# Ordinal encoding
df['tod_ordinal'] = df['Time_of_day_category'].map(TOD_ORDER).fillna(-1).astype(int)

# Cyclic encoding — handles midnight wrap (hour 23 → hour 0)
hour_valid = df['hour'].clip(lower=0)
df['hour_sin'] = np.sin(2 * np.pi * hour_valid / 24)
df['hour_cos'] = np.cos(2 * np.pi * hour_valid / 24)

# Binary time-window flags
df['is_early_morning']  = ((df['hour'] >= 5)  & (df['hour'] < 9)).astype(int)
df['is_morning']        = ((df['hour'] >= 9)  & (df['hour'] < 12)).astype(int)
df['is_early_afternoon']= ((df['hour'] >= 12) & (df['hour'] < 15)).astype(int)
df['is_late_afternoon'] = ((df['hour'] >= 15) & (df['hour'] < 18)).astype(int)
df['is_evening']        = ((df['hour'] >= 18) & (df['hour'] < 22)).astype(int)
df['is_night']          = ((df['hour'] >= 22) | (df['hour'] < 5)).astype(int)

print("Time-of-day features added:")
print(df[['hour','tod_ordinal','hour_sin','hour_cos','is_morning','is_night']].head(3))


In [ ]:
# Visualise cyclic encoding
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

hours = np.arange(24)
axes[0].plot(hours, np.sin(2*np.pi*hours/24), color=BLUE, lw=2, label='sin')
axes[0].plot(hours, np.cos(2*np.pi*hours/24), color='#E67E22', lw=2, label='cos')
axes[0].set_title('Cyclic Hour Encoding', fontweight='bold')
axes[0].set_xlabel('Hour'); axes[0].legend()
axes[0].axvline(0, color='grey', linestyle=':')
axes[0].axvline(23, color='grey', linestyle=':')
axes[0].annotate('Midnight continuity', xy=(23, 0), xytext=(16, 0.5),
                  arrowprops=dict(arrowstyle='->', color='grey'), fontsize=8)

# Acceptance rate per hour
hr_acc = df[df['hour']>=0].groupby('hour')['label'].mean()
axes[1].bar(hr_acc.index, hr_acc.values*100, color=BLUE, alpha=0.8)
axes[1].axhline(50, color='red', linestyle='--', linewidth=0.8)
axes[1].set_title('Acceptance Rate by Hour', fontweight='bold')
axes[1].set_xlabel('Hour of Day'); axes[1].set_ylabel('Acceptance %')
plt.tight_layout(); plt.show()


## B. Calendar Features

In [ ]:
SEASON_ORDER = {'Spring': 0, 'Summer': 1, 'Autumn': 2, 'Winter': 3}

# Year offset (avoids spurious ordinality of raw year values)
df['years_since_2009'] = (df['year'] - 2009).clip(lower=0)

# Cyclic month (Dec=12 → Jan=1 are adjacent)
month_valid = df['month'].clip(lower=1)
df['month_sin'] = np.sin(2 * np.pi * month_valid / 12)
df['month_cos'] = np.cos(2 * np.pi * month_valid / 12)

# Cyclic day-of-week (Sun=6 → Mon=0 are adjacent)
dow_valid = df['day_of_week'].fillna(0)
df['dow_sin'] = np.sin(2 * np.pi * dow_valid / 7)
df['dow_cos'] = np.cos(2 * np.pi * dow_valid / 7)

# Season ordinal
df['season_ordinal'] = df['Season'].map(SEASON_ORDER).fillna(-1).astype(int)

# Political boundary days
df['is_monday'] = (df['day_of_week'] == 0).astype(int)
df['is_friday'] = (df['day_of_week'] == 4).astype(int)

print("Calendar features added ✓")
print(df[['year','years_since_2009','month_sin','month_cos','season_ordinal']].head(3))


## C. Meeting Metadata

In [ ]:
# is_plenary (strongest single predictor)
df['is_plenary'] = (df['Meeting_type'] == 'Plenair').astype(int)

# Log-transform right-skewed duration and density
df['log_duration'] = np.log1p(df['Topic_duration_minutes'])
df['log_density']  = np.log1p(df['Topic_density_per_day'])

# High-density day flag (top quartile)
q75 = df['Topic_density_per_day'].quantile(0.75)
df['high_density_day'] = (df['Topic_density_per_day'] >= q75).astype(int)

# Quick correlation check
print("Pearson r with label:")
for col in ['is_plenary','log_duration','log_density','high_density_day']:
    r = df[[col,'label']].corr().iloc[0,1]
    print(f"  {col:25s}: r = {r:+.4f}")


## D. Topic Category One-Hot

In [ ]:
CATEGORY_LIST = [
    'Governance & Administration', 'Infrastructure & Transport',
    'Economy & Finance', 'Healthcare', 'Education', 'Housing',
    'Public Safety & Policing', 'Employment & Labour',
    'Environment & Climate', 'Social Welfare',
]

for cat in CATEGORY_LIST:
    col = 'cat_' + cat.lower().replace(' ', '_').replace('&', 'and')
    df[col] = (df['Topic_category'] == cat).astype(int)

print(f"Category columns added: {[c for c in df.columns if c.startswith('cat_')]}")


## E. Interaction Features

In [ ]:
# Plenary × time interactions
df['plenary_x_tod']        = df['is_plenary'] * df['tod_ordinal'].clip(lower=0)
df['plenary_x_night']      = df['is_plenary'] * df['is_night']
df['high_density_x_night'] = df['high_density_day'] * df['is_night']

# Duration × density (long AND busy sessions)
df['duration_x_density']   = df['log_duration'] * df['log_density']

print("Interaction features added ✓")
print(df[['plenary_x_tod','plenary_x_night','high_density_x_night','duration_x_density']].describe())


## Correlation Matrix — All Structured Features

In [ ]:
STRUCTURED_COLS = [
    'hour', 'tod_ordinal', 'hour_sin', 'hour_cos',
    'is_early_morning', 'is_morning', 'is_early_afternoon',
    'is_late_afternoon', 'is_evening', 'is_night',
    'years_since_2009', 'month_sin', 'month_cos',
    'dow_sin', 'dow_cos', 'season_ordinal', 'is_monday', 'is_friday',
    'is_plenary', 'log_duration', 'log_density', 'high_density_day',
    'plenary_x_tod', 'plenary_x_night', 'high_density_x_night',
    'duration_x_density',
    'cat_governance_and_administration', 'cat_infrastructure_and_transport',
    'cat_economy_and_finance', 'cat_healthcare', 'cat_education',
    'cat_housing', 'cat_public_safety_and_policing',
    'cat_employment_and_labour', 'cat_environment_and_climate',
    'cat_social_welfare',
]

# Correlation with target
corr = df[STRUCTURED_COLS + ['label']].corr()['label'].drop('label').sort_values()

fig, ax = plt.subplots(figsize=(6, 11))
colors = ['#C0392B' if v < 0 else '#27AE60' for v in corr.values]
ax.barh(corr.index, corr.values, color=colors, edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', linewidth=0.7)
ax.set_title("Feature Correlation with 'label'\n(green=positive, red=negative)",
             fontweight='bold')
ax.set_xlabel("Pearson r")
plt.tight_layout(); plt.show()

print(f"\nTop 5 positive: {corr.tail(5).index.tolist()}")
print(f"Top 5 negative: {corr.head(5).index.tolist()}")


## Save Features

In [ ]:
OUTPUT_PATH = "features.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df)} rows with {len(df.columns)} columns → {OUTPUT_PATH}")
print(f"Structured feature columns: {len(STRUCTURED_COLS)}")
